In [1]:
%pip install google-cloud-storage
%pip install numpy
%pip install torch
%pip install transformers
%load_ext autoreload
%autoreload 2
%pip install accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install opensmile soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 93.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [opensmile]13 [audformat]g]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**pulling meld audio data**

In [1]:
import opensmile
import pandas as pd
import io
from google.cloud import storage
from tqdm import tqdm
import soundfile as sf
import numpy as np
import tempfile
import subprocess
from pathlib import Path

client = storage.Client(project="gen-lang-client-0105254213")
bucket = client.bucket("meld")
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

/home/liaojd/SenticCrystal/meld_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
# lists all top level folders
blobs = bucket.list_blobs()
prefixes = set()
for blob in blobs:
    top = blob.name.split("/")[0]
    prefixes.add(top)
    
print(prefixes)

{'wav2vec2_features', 'output_repeated_splits_test', 'dev_sent_emo.csv', 'train', 'dev_splits_complete', 'README.txt', 'hubert_features', 'embeddings', 'test_sent_emo.csv'}


In [ ]:
def process_blob(blob):
    audio_bytes = blob.download_as_bytes()
    
    tmp_mp4 = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    tmp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    
    try:
        tmp_mp4.write(audio_bytes)
        tmp_mp4.flush()
        tmp_mp4.close()
        tmp_wav.close()
        
        result = subprocess.run(
            ["ffmpeg", "-y", "-i", tmp_mp4.name,
             "-ar", "16000", "-ac", "1", tmp_wav.name],
            capture_output=True, timeout=30
        )
        
        if result.returncode != 0:
            print(f"ffmpeg stderr: {result.stderr.decode()}")
            raise RuntimeError(f"ffmpeg failed: {result.stderr.decode()}")
        
        audio, sr = sf.read(tmp_wav.name)
    finally:
        os.unlink(tmp_mp4.name)
        os.unlink(tmp_wav.name)
    
    feats = smile.process_signal(audio, sr)
    row = feats.iloc[0].to_dict()
    row["filename"] = Path(blob.name).stem
    return row

splits = {
    "train": "train/",
    "dev":   "dev_splits_complete/",
    "test":  "output_repeated_splits_test/",
}

records = []
for split_name, prefix in splits.items():
    blobs = list(bucket.list_blobs(prefix=prefix))
    for blob in tqdm(blobs, desc=split_name):
        if blob.name.endswith((".wav", ".mp4")):
            try:
                row = process_blob(blob)
                row["split"] = split_name
                records.append(row)
            except Exception as e:
                print(f"✗ {blob.name}: {e}")

# Save results
df = pd.DataFrame(records)
df.to_csv("meld_egemaps_raw.csv", index=False)
print(f"Done → {len(df)} rows × {df.shape[1]} cols")
print(df.head())

train:   7%|▋         | 674/9990 [03:44<48:03,  3.23it/s]  

✗ train/train_splits/dia125_utt3.mp4: Command '['ffmpeg', '-y', '-i', '/tmp/tmpqchhd_wm.mp4', '-ar', '16000', '-ac', '1', '/tmp/tmpvarikqc3.wav']' returned non-zero exit status 1.


train:  11%|█         | 1095/9990 [06:04<50:21,  2.94it/s] 

In [2]:
import pandas as pd

df = pd.read_csv("/home/liaojd/SenticCrystal/scripts/MELD/opensmile/meld_egemaps_raw.csv")

# Basic checks
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1]}")
print(f"\nSplit counts:")
print(df["split"].value_counts())

# Check for missing values
print(f"\nMissing values per column:")
print(df.isnull().sum().sum(), "total nulls")

# Preview feature columns
print(f"\nSample row:")
print(df.head(1))

Total rows: 13839
Total columns: 90

Split counts:
split
train    9980
test     2747
dev      1112
Name: count, dtype: int64

Missing values per column:
0 total nulls

Sample row:
   F0semitoneFrom27.5Hz_sma3nz_amean  F0semitoneFrom27.5Hz_sma3nz_stddevNorm  \
0                          26.082586                                0.095447   

   F0semitoneFrom27.5Hz_sma3nz_percentile20.0  \
0                                    23.94496   

   F0semitoneFrom27.5Hz_sma3nz_percentile50.0  \
0                                   26.101959   

   F0semitoneFrom27.5Hz_sma3nz_percentile80.0  \
0                                   27.402922   

   F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2  \
0                                  3.457962   

   F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope  \
0                                   137.764526   

   F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope  \
0                                      90.810478   

   F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope  \
0         